# Build Your First Voice Agent using LiveKit

This cookbook is a beginner-friendly, hands-on walkthrough for building a **real-time, multilingual voice agent** with **[LiveKit Agents](https://docs.livekit.io/agents/)** and **[Sarvam AI](https://docs.sarvam.ai)**.

By the end of this notebook you will have a voice agent that can:

- **Listen** to a caller speaking any of 10 Indian languages (or English)
- **Understand** the request with a Sarvam chat model
- **Respond** back in a natural-sounding voice

LiveKit handles the real-time audio transport (WebRTC rooms), while Sarvam AI provides the speech-to-text (**Saaras**), the LLM (**sarvam-30b** / **sarvam-105b**), and the text-to-speech (**Bulbul**) — all wired together through the official `livekit-plugins-sarvam` plugin.

> **Note on running this notebook:** A voice agent is a long-running, real-time process (it waits for someone to join a LiveKit room), so it is not something you can "run to completion" in a single notebook cell. Instead, this notebook builds the agent file for you (`agent.py`) using `%%writefile`, and then shows you the exact terminal commands to run and test it. Treat the code cells as *generators* for a project you'll run from your terminal.

## Overview

| | |
|---|---|
| **Telephony / transport** | [LiveKit Cloud](https://cloud.livekit.io) (free tier available) |
| **Speech-to-text** | Sarvam **Saaras v3** (`saaras:v3`) |
| **LLM** | Sarvam chat models (`sarvam-30b`, `sarvam-105b`) |
| **Text-to-speech** | Sarvam **Bulbul v3** (`bulbul:v3`) |
| **Languages** | 10 Indian languages + English, with auto-detect |
| **Lines of code** | ~40 |

### Quick overview of the steps

1. Get API keys (LiveKit, Sarvam).
2. Install `livekit-agents` with the `sarvam` and `silero` extras.
3. Create a `.env` file with your credentials.
4. Write the agent (`agent.py`).
5. Run it in dev mode and talk to it from the LiveKit console.

## Prerequisites

- Python 3.9 or higher.
- A [LiveKit Cloud](https://cloud.livekit.io) account (free) — you'll need the project **URL**, **API key**, and **API secret**.
- A [Sarvam AI](https://dashboard.sarvam.ai) API key.

## 1. Install dependencies

The `sarvam` extra pulls in the Sarvam STT/LLM/TTS plugin; `silero` provides the local voice-activity-detection (VAD) model used elsewhere in the LiveKit ecosystem. `python-dotenv` loads your `.env` file at runtime.

In [ ]:
%pip install -Uqq "livekit-agents[sarvam,silero]" python-dotenv

## 2. Configure your credentials

Get your keys from:

- **Sarvam AI** → [dashboard.sarvam.ai](https://dashboard.sarvam.ai)
- **LiveKit Cloud** → [cloud.livekit.io](https://cloud.livekit.io) → your project's **Settings**

Never commit real API keys to version control. The cell below writes a `.env.example` template — copy it to `.env` and fill in your real values locally:

```bash
cp .env.example .env
```

In [ ]:
%%writefile .env.example
# Copy this file to `.env` and fill in your real credentials.
# Do not commit `.env` to version control.

LIVEKIT_URL=wss://your-project-xxxxx.livekit.cloud
LIVEKIT_API_KEY=your_livekit_api_key
LIVEKIT_API_SECRET=your_livekit_api_secret
SARVAM_API_KEY=your_sarvam_api_key

## 3. Write the voice agent

The agent below defines a `VoiceAgent` with three Sarvam components wired in:

- `stt=sarvam.STT(...)` — Saaras v3 converts the caller's speech to text.
- `llm=sarvam.LLM(...)` — a Sarvam chat model generates the reply.
- `tts=sarvam.TTS(...)` — Bulbul v3 converts the reply back to speech.

`%%writefile` saves this cell's contents to `agent.py` in the current directory so you can run it from a terminal in the next step.

In [ ]:
%%writefile agent.py
import logging

from dotenv import load_dotenv
from livekit.agents import JobContext, WorkerOptions, cli
from livekit.agents.voice import Agent, AgentSession
from livekit.plugins import sarvam

# Load LIVEKIT_* and SARVAM_API_KEY from .env
load_dotenv()

logger = logging.getLogger("voice-agent")
logger.setLevel(logging.INFO)


class VoiceAgent(Agent):
    """A multilingual voice assistant built on Sarvam's STT, LLM, and TTS."""

    def __init__(self) -> None:
        super().__init__(
            instructions="""
                You are a helpful voice assistant.
                Be friendly, concise, and conversational.
                Speak naturally as if you're having a real conversation.
            """,
            # Saaras v3 STT - converts speech to text.
            stt=sarvam.STT(
                language="unknown",  # auto-detect language, or set "en-IN", "hi-IN", etc.
                model="saaras:v3",
                mode="transcribe",
            ),
            # Sarvam LLM - the "brain" that processes and generates responses.
            llm=sarvam.LLM(model="sarvam-30b"),
            # Bulbul v3 TTS - converts text to speech.
            tts=sarvam.TTS(
                target_language_code="en-IN",
                model="bulbul:v3",
                speaker="shubh",  # female: priya, simran, ishita, kavya | male: aditya, anand, rohan
            ),
        )

    async def on_enter(self) -> None:
        """Called when a user joins the room; the agent speaks first."""
        self.session.generate_reply()


async def entrypoint(ctx: JobContext) -> None:
    """Main entry point - LiveKit calls this when a user connects to a room."""
    logger.info("User connected to room: %s", ctx.room.name)

    session = AgentSession()
    await session.start(agent=VoiceAgent(), room=ctx.room)


if __name__ == "__main__":
    cli.run_app(WorkerOptions(entrypoint_fnc=entrypoint))

## 4. Run and test your agent

From a terminal, in the same directory as `agent.py` and `.env`:

```bash
# Start the agent worker (connects to LiveKit Cloud and waits for a room)
python agent.py dev
```

In a **second** terminal, start the interactive console client to talk to it directly from your machine's microphone/speakers:

```bash
python agent.py console
```

That's it — you've built your first real-time, multilingual voice agent!

## Customization examples

The snippets below show how to adapt the `stt=`/`tts=`/`llm=` arguments inside `VoiceAgent.__init__` for different scenarios. Swap the corresponding block into `agent.py` and re-run `python agent.py dev`.

### Hindi voice agent

```python
stt=sarvam.STT(language="hi-IN", model="saaras:v3", mode="transcribe"),
tts=sarvam.TTS(
    target_language_code="hi-IN",
    model="bulbul:v3",
    speaker="simran",  # or: priya, ishita, kavya, aditya, anand, rohan
),
```

### Tamil voice agent

```python
stt=sarvam.STT(language="ta-IN", model="saaras:v3", mode="transcribe"),
tts=sarvam.TTS(target_language_code="ta-IN", model="bulbul:v3", speaker="shubh"),
```

### Multilingual agent (auto-detect)

```python
stt=sarvam.STT(language="unknown", model="saaras:v3", mode="transcribe"),  # auto-detects language
tts=sarvam.TTS(target_language_code="en-IN", model="bulbul:v3", speaker="anand"),
```

### Speech-to-English agent (Saaras `translate` mode)

Saaras v3 handles both transcription (same-language output) and translation (English output) via the `mode` parameter. Use `mode="translate"` when the caller speaks an Indian language but you want the LLM to process and respond in English — Saaras auto-detects the source language and translates it directly to English text.

```python
# Caller speaks Hindi -> Saaras converts to English -> LLM processes -> replies in English
stt=sarvam.STT(model="saaras:v3", mode="translate"),
llm=sarvam.LLM(model="sarvam-30b"),
tts=sarvam.TTS(target_language_code="en-IN", model="bulbul:v3", speaker="aditya"),
```

## Available options

### Language codes

| Language | Code |
|---|---|
| English (India) | `en-IN` |
| Hindi | `hi-IN` |
| Bengali | `bn-IN` |
| Tamil | `ta-IN` |
| Telugu | `te-IN` |
| Gujarati | `gu-IN` |
| Kannada | `kn-IN` |
| Malayalam | `ml-IN` |
| Marathi | `mr-IN` |
| Punjabi | `pa-IN` |
| Odia | `od-IN` |
| Auto-detect | `unknown` |

### Speaker voices (Bulbul v3)

- **Male (23):** shubh (default), aditya, rahul, rohan, amit, dev, ratan, varun, manan, sumit, kabir, aayan, ashutosh, advait, anand, tarun, sunny, mani, gokul, vijay, mohit, rehan, soham
- **Female (14):** ritu, priya, neha, pooja, simran, kavya, ishita, shreya, roopa, tanya, shruti, suhani, kavitha, rupali

In [ ]:
# Quick reference you can import/reuse elsewhere: language codes supported by
# Sarvam STT/TTS in this integration, and a couple of speaker voices per gender.
SUPPORTED_LANGUAGES = {
    "English (India)": "en-IN",
    "Hindi": "hi-IN",
    "Bengali": "bn-IN",
    "Tamil": "ta-IN",
    "Telugu": "te-IN",
    "Gujarati": "gu-IN",
    "Kannada": "kn-IN",
    "Malayalam": "ml-IN",
    "Marathi": "mr-IN",
    "Punjabi": "pa-IN",
    "Odia": "od-IN",
    "Auto-detect": "unknown",
}

for language, code_ in SUPPORTED_LANGUAGES.items():
    print(f"{language:<20} {code_}")

## Best practices

Follow these recommendations when using the Sarvam plugin with LiveKit for the best turn-taking and latency behavior.

### 1. Do not pass `vad` to `AgentSession`

Voice activity detection is handled internally by the Sarvam plugin.

```python
# Avoid this
session = AgentSession(vad=silero.VAD.load())

# Do this instead
session = AgentSession()
```

### 2. Enable the flush signal in STT

`flush_signal=True` lets the plugin emit start/end-of-speech events, which is essential for proper turn-taking.

```python
stt = sarvam.STT(
    language="unknown",
    model="saaras:v3",
    mode="transcribe",
    flush_signal=True,
)
```

### 3. Set turn detection to `"stt"`

This ensures turn detection is driven by the Sarvam plugin's own speech start/end signals.

```python
session = AgentSession(turn_detection="stt")
```

### 4. Configure `min_endpointing_delay`

The Sarvam STT plugin has a processing latency of roughly 70ms. Setting `min_endpointing_delay=0.07` lets the agent move to the LLM step as soon as STT finishes, minimizing response delay.

```python
session = AgentSession(turn_detection="stt", min_endpointing_delay=0.07)
```

### Putting it all together

```python
stt = sarvam.STT(
    language="unknown",
    model="saaras:v3",
    mode="transcribe",
    flush_signal=True,
)

session = AgentSession(turn_detection="stt", min_endpointing_delay=0.07)
```

## Troubleshooting

- **API key errors** — Check that all keys are in your `.env` file and that the file is in the same directory as `agent.py`.
- **Module not found** — Re-run the install cell; on some platforms you may need to install `livekit-agents[sarvam,silero]` inside quotes (`zsh` treats `[...]` as a glob).
- **Poor transcription** — Try `language="unknown"` for auto-detection, or set the exact language code (`en-IN`, `hi-IN`, etc.) if you already know it.

## Additional resources

- [Sarvam AI Documentation](https://docs.sarvam.ai)
- [LiveKit Documentation](https://docs.livekit.io)
- [LiveKit Sarvam LLM Plugin](https://docs.livekit.io/agents/models/llm/sarvam/)
- [LiveKit Sarvam STT Plugin](https://docs.livekit.io/agents/models/stt/plugins/sarvam/)
- [LiveKit Sarvam TTS Plugin](https://docs.livekit.io/agents/models/tts/plugins/sarvam/)

## Need help?

- Sarvam Support: developer@sarvam.ai
- Community: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)